# Live Inference

Streams sensor data from the Arduino over USB serial and runs the PyTorch model in real-time.

**Before running:** Set `COM_PORT` to the correct serial port for your BLE Sense board.

In [ ]:
# %pip install pyserial torch numpy


In [ ]:
import threading
import time

import numpy as np
import torch
import torch.nn.functional as F
import serial

from emorec import EmotionCNN, SERIAL_COLS, FEATURE_COLS, WINDOW_SIZE, STEP_SIZE

## 1. Load Model

In [ ]:
MODEL_PATH = 'Ml-Models/emotionv2_tiny_vgg_updated_windowing_14features.pth'

device = torch.device('cpu')

model = EmotionCNN(input_channels=14, hidden_units=10, num_classes=4)
model.load_state_dict(torch.load(MODEL_PATH, weights_only=True, map_location=device))
model.eval()
print('Model loaded.')

## 2. Serial Configuration

In [ ]:
COM_PORT = '/dev/cu.usbmodem21201'  # <-- update this

# Indices of FEATURE_COLS within the SERIAL_COLS stream (used to drop orientation columns)
FEATURE_INDICES = [SERIAL_COLS.index(c) for c in FEATURE_COLS]

CLASS_LABELS = {0: 'Distracted', 1: 'Focused', 2: 'Relaxed', 3: 'Stressed'}

## 3. Live Streaming + Inference

In [ ]:
live_buffer = []
buffer_lock = threading.Lock()


def data_collection_thread():
    ser = serial.Serial(COM_PORT, 115200)
    ser.reset_input_buffer()
    print('Listening to Arduino...')
    while True:
        try:
            raw = ser.readline().decode('utf-8').strip()
            parts = raw.split(',')
            if len(parts) != len(SERIAL_COLS):
                continue
            sample = [float(parts[i]) for i in FEATURE_INDICES]
            with buffer_lock:
                live_buffer.append(sample)
        except Exception:
            pass


def process_window(data_list):
    """Normalise a list of raw samples and return (predicted_class, confidence)."""
    data = np.array(data_list, dtype=np.float32)
    windows = []
    for i in range(0, len(data) - WINDOW_SIZE + 1, STEP_SIZE):
        w = data[i : i + WINDOW_SIZE]
        mean = w.mean(axis=0, keepdims=True)
        std  = w.std(axis=0,  keepdims=True)
        windows.append((w - mean) / (std + 1e-8))
    if not windows:
        return None, 0.0
    x = torch.tensor(np.stack(windows), dtype=torch.float32).transpose(1, 2)
    with torch.no_grad():
        probs = F.softmax(model(x), dim=1).mean(dim=0)
        conf, pred = probs.max(dim=0)
    return pred.item(), conf.item()


def inference_thread():
    LONG_WINDOW = WINDOW_SIZE * 12   # ~60 seconds of context
    INFER_INTERVAL = 5.0             # run inference every N seconds
    last_inference = time.time()

    print(f'Waiting for {LONG_WINDOW} samples...')
    while True:
        with buffer_lock:
            buf_len = len(live_buffer)
        if buf_len < LONG_WINDOW:
            time.sleep(0.1)
            continue
        if time.time() - last_inference < INFER_INTERVAL:
            time.sleep(0.05)
            continue
        last_inference = time.time()

        with buffer_lock:
            snapshot = list(live_buffer)

        pred, conf = process_window(snapshot[-WINDOW_SIZE:])
        print(f'[{time.strftime("%H:%M:%S")}]  {CLASS_LABELS.get(pred, "?")}  ({conf:.1%})')


threading.Thread(target=data_collection_thread, daemon=True).start()
threading.Thread(target=inference_thread,        daemon=True).start()

try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    print('Stopped.')